# Stage 2 — SAW Recommender

**What this notebook does:** combines the two signals — *weak now* (Stage 0) and *will worsen* (Stage 1) — and uses **Simple Additive Weighting (SAW)** to rank the 5 cognitive domains for training, with transparent reasons. A separate **adapter** (built later in this notebook) routes each patient to the **AD arm**, **LBD arm**, or a **safe fallback**, based on the diagnosis probabilities from Components 1–3.

**Plan:** 1) build the criteria table → 2) SAW scoring (normalise → weight → rank → reasons) → 3) adapter + mock + safe fallback → 4) end-to-end → 5) evaluation (with/without decline, vs baseline).

**This step (Step 1):** assemble the per-patient **criteria table** — the inputs SAW will score. No scoring yet.

## Setup

Load the two signals saved by the earlier stages, and set the weakness threshold.

The **weakness threshold is a cited clinical convention** a domain is "weak" when `z ≤ −1.0 SD`. We never tune it to improve results; we will instead run a sensitivity analysis (−1.0 / −1.5 / −2.0) later.

In [ ]:
import os, json
import numpy as np, pandas as pd

PROJ = r"D:\SLIIT\Final Research\NeuroSage-cognitive\cognitive-train-system"
dom = ["memory", "attention", "visuospatial", "executive", "language"]

# CITED clinical convention (CLAUDE_threshold_v1.md) -- DO NOT tune/auto-optimise.
# NIA-AA 1.0-1.5 SD; Heaton 1 SD; MoCA-Tamil < -1 SD.  z <= -1.0 flags a weak domain.
WEAK_THRESHOLD = -1.0

prof = pd.read_parquet(os.path.join(PROJ, "data", "cognitive_profiles.parquet"))      # weak-now: domain z-scores
decl = pd.read_parquet(os.path.join(PROJ, "data", "decline_predictions_oof.parquet"))  # will-worsen: OOF severities (leak-free)
print("profiles:", prof.shape, "| decline (out-of-fold):", decl.shape)

## Step 1 — Build the SAW criteria table

For each patient and each of the 5 domains we assemble the three inputs SAW will use:

- **weak-now severity** = `−z` — higher means more impaired *right now* (from Stage 0).
- **is_weak** = `z ≤ −1.0` — the cited weakness flag. We use it as a **criterion** (option B), so a domain predicted to *worsen into* weakness can still surface.
- **will-worsen severity** = the Stage 1 prediction — higher means faster predicted decline. We use the **out-of-fold** values so any later evaluation is leakage-free.

We keep every profiled patient. The will-worsen columns are **blank for patients without enough longitudinal data** (one visit, or no two visits that overlap on a domain) — SAW handles that in the next step. The two severities are on **different scales** (weak-now can be large; will-worsen ≈ 0–1.7), so we leave them **raw** here; SAW normalises them **within each patient** at the scoring step.

**What the weak-% table means:** it is **descriptive** — how common the flag is in each arm — **not** proof of clinical optimality or of "disease signatures" (those claims need separate clinical validation). Also note for the methodology: z-scores use the **NACC cognitively-normal participants in this dataset, without age/education adjustment**.

In [ ]:
C = prof[["NACCID", "diagnosis"]].copy()
for d in dom:
    C["weaknow_" + d] = -prof[d]                      # higher = weaker now
    C["isweak_" + d]  = prof[d] <= WEAK_THRESHOLD     # cited -1.0 SD flag (as a criterion)

C = C.merge(decl[["NACCID"] + ["severity_" + d for d in dom]], on="NACCID", how="left")
C = C.rename(columns={"severity_" + d: "willworsen_" + d for d in dom})

OUT = os.path.join(PROJ, "data", "saw_criteria.parquet")
C.to_parquet(OUT, index=False)
print("Saved:", OUT)

has_decline = C[["willworsen_" + d for d in dom]].notna().any(axis=1)
print(f"\nPatients: {len(C):,} total | with a will-worsen signal: {has_decline.sum():,}")

print("\n% of patients with each domain flagged WEAK (z <= -1.0), by arm:")
flag = C.groupby("diagnosis")[["isweak_" + d for d in dom]].mean().mul(100).round(0)
flag.columns = dom
print(flag.to_string())

print("\nPreview (memory & visuospatial columns):")
cols = ["NACCID", "diagnosis", "weaknow_memory", "isweak_memory", "willworsen_memory",
        "weaknow_visuospatial", "isweak_visuospatial", "willworsen_visuospatial"]
print(C.head(4)[cols].round(2).to_string(index=False))